# DNA Complement and EcoRI Restriction Site Finder

Two small sequence-analysis exercises in pure Python:

1. Build the **complement** and the **reverse complement** of a DNA sequence.
2. Find every **EcoRI restriction site** (`GAATTC`) in a sequence, report 1-based positions, and work out the fragment sizes a digest would give.

**Note:** the sequence in part 2 is a short made-up sample, not a real plasmid. For real work, download a sequence from NCBI.

## 1. Complement and reverse complement

A DNA base pairs with its partner: A with T, and G with C. The partner strand runs in the opposite direction, so the real "other strand", written 5'->3', is the **reverse complement**: complement every base, then reverse the result.

In [1]:
COMPLEMENT = {"A": "T", "T": "A", "G": "C", "C": "G"}

def complement(dna):
    """Return the complement of a DNA sequence (read in the same direction)."""
    return "".join(COMPLEMENT[base] for base in dna.upper())

def reverse_complement(dna):
    """Return the reverse complement (the opposite strand, read 5' -> 3')."""
    return complement(dna)[::-1]

dna = "ATGCTCATCCATTG"
print("DNA sequence:        ", dna)
print("Complement:          ", complement(dna))
print("Reverse complement:  ", reverse_complement(dna))

DNA sequence:         ATGCTCATCCATTG
Complement:           TACGAGTAGGTAAC
Reverse complement:   CAATGGATGAGCAT


## 2. Finding EcoRI restriction sites

EcoRI is a restriction enzyme that recognises `GAATTC` and cuts between the G and the A (`G^AATTC`). Knowing where it cuts is the first step in planning a digest or a cloning experiment.

`GAATTC` reads the same on both strands (it is its own reverse complement), so searching one strand finds every site.

The function below uses a loop so that it finds **all** sites, not only the first one. Positions are **1-based**, the usual convention in sequence analysis.

In [2]:
sample_seq = (
    "TTTCCTCATGCAATTCAAAACCATGTCCGTAATGTAGGCGAAATAGTAAACCATTTTACG"
    "GAATTCGAGGATACCAAATTCCTCCTTATTCAGGACCTAACCTGAGGTAAACCAGGTCTC"
    "TGAATTCCCGCCCCCTTATAAAAGCTGTTGCACCTAGCCAAGTTCAACGGCAGCTG"
)

print("Sequence length:", len(sample_seq), "bp")

Sequence length: 176 bp


In [3]:
def find_sites(seq, site="GAATTC"):
    """Return the 1-based start position of every occurrence of `site` in `seq`."""
    seq = seq.upper()
    positions = []
    for i in range(len(seq) - len(site) + 1):
        if seq[i:i + len(site)] == site:
            positions.append(i + 1)          # +1 converts to 1-based
    return positions

sites = find_sites(sample_seq)

if sites:
    print(f"EcoRI sites found: {len(sites)}")
    for s in sites:
        print(f"  site starts at position {s}")
else:
    print("No EcoRI sites found")

EcoRI sites found: 2
  site starts at position 61
  site starts at position 122


### Fragment sizes after a digest

EcoRI cuts after the first base of the site (`G^AATTC`). For a **linear** sequence, the cut points split it into fragments. The first cut falls after position `s` (the G), so each fragment length is the distance between consecutive cut points.

In [4]:
def fragment_sizes(seq, site="GAATTC", cut_offset=1):
    """Fragment lengths from digesting a linear sequence.

    cut_offset = number of bases from the start of the site to the cut
    (1 for EcoRI, which cuts G^AATTC).
    """
    cuts = [s - 1 + cut_offset for s in find_sites(seq, site)]   # cut after this base
    edges = [0] + cuts + [len(seq)]
    return [edges[i + 1] - edges[i] for i in range(len(edges) - 1)]

sizes = fragment_sizes(sample_seq)
print("Fragment sizes (bp):", sizes)
print("Total:", sum(sizes), "bp  (should equal the sequence length)")

Fragment sizes (bp): [61, 61, 54]
Total: 176 bp  (should equal the sequence length)


## What I learned

- Complementing a sequence is not the same as reverse-complementing it, and the reverse complement is the true opposite strand
- Python's `str.find()` returns a 0-based index and only the first match, so a loop is needed to find every site
- Converting between 0-based (Python) and 1-based (biology) positions
- Turning cut positions into expected fragment sizes, which can be checked against a gel

## Possible improvements
- Add other enzymes (BamHI `GGATCC`, HindIII `AAGCTT`) by passing a different `site`
- Handle circular plasmids, where the last and first fragments join
- Compare the results with Biopython's `Bio.Restriction` module